In [ ]:
import glob # For file pattern matching
import os # For handling file paths
# Define the directory containing the probe data
PROBES_DIR = os.path.join("C:\\Users\\陳懷浯\\OneDrive\\桌面\\biogas plant\\SAMPLE O")
print(f"PROBES_DIR: {PROBES_DIR}")
print(f"os.path.basename(PROBES_DIR): {os.path.basename(PROBES_DIR)}")


In [ ]:
glob.glob(os.path.join(PROBES_DIR, "Sensor 4*.csv"))[:10] 
# List the first 10 CSV files in the PROBES_DIR that start with "Sensor 4"

In [ ]:
import pandas as pd
import numpy as np

csv_path = os.path.join(PROBES_DIR, "Sensor 1 Sample O.1 10 MIN.csv") # Path to the specific CSV file to read
print(f"csv_path basename: {os.path.basename(csv_path)}") # Print the base name of the CSV file
print(f"csv_path dirname: {os.path.dirname(csv_path)}") # Print the directory name of the CSV file
print(os.path.splitext(os.path.basename(csv_path)))# Print the base name and extension of the CSV file
df = pd.read_csv(csv_path)# Read the CSV file into a DataFrame

df.head()# Display the first few rows of the DataFrame

In [ ]:
dfd = df.drop(columns=df.columns[0], inplace=False)
# Drop the first column of the DataFrame, which is often an index or timestamp
dfd.head()# Display the first few rows of the modified DataFrame without the first column

In [ ]:
df.index.values# Display the index values of the DataFrame, which may represent timestamps or row numbers

In [ ]:
wl_range = df.columns[1:].values# Get the column names starting from the second column, which may represent wavelength ranges
wl_range = np.array(wl_range, dtype=int)# Convert the column names to integers, assuming they represent wavelength ranges
wl_range

### Data Loading

#### Read and process csv

In [ ]:
def read_csv_data(path):
    df = pd.read_csv(path, encoding='ISO-8859-1')# Read the CSV file into a DataFrame with the specified encoding
    # Modification to the df
    df.drop(columns=df.columns[0], inplace=True)
    # average the intensity of the same wavelength
    wavelength = df.columns.to_numpy(dtype=int)
    intensity = df.mean(axis=0).to_numpy(dtype=float)
    return wavelength, intensity

# TODO: deal with empty cuvette, lamp on/off, etc.
def get_csv_paths(probes_dir, sensor, reference_sample: bool = False):
    if reference_sample:
        patterns = [
            os.path.join(probes_dir, f"{sensor}_lamp_on.csv"),
            os.path.join(probes_dir, f"{sensor}_lamp_off.csv"),
        ]
        paths = []
        for pattern in patterns:
            paths.extend(glob.glob(pattern))
        return paths
    else:
        pattern = os.path.join(probes_dir, f"{sensor} Sample*.csv")
        return glob.glob(pattern)

In [ ]:
wavelength, intensity = read_csv_data(os.path.join(PROBES_DIR, "Sensor 4 Sample P.1 Baseline.csv"))
print(f"Wavelength length of sensor 4: {len(wavelength)}\n")# Print the length of the wavelength array for sensor 4
print(f"Intensity length: {len(intensity)}")

In [ ]:
print(f"First 5 wavelength/intensity pairs:")
for i in range(min(5, len(wavelength))):# Loop through the first 5 indices (or fewer if there are less than 5) and print the wavelength and intensity pairs, formatted to 2 decimal places
    print(f"  WL: {wavelength[i]}, Intensity: {intensity[i]:.2f}")# Print the first 5 wavelength and intensity pairs, formatted to 2 decimal places

print(f"Last 5 wavelength/intensity pairs:")
for i in range(max(0, len(wavelength) - 5), len(wavelength)):# Loop through the last 5 indices (or fewer if there are less than 5) and print the wavelength and intensity pairs, formatted to 2 decimal places
    print(f"  WL: {wavelength[i]}, Intensity: {intensity[i]:.2f}")# Print the last 5 wavelength and intensity pairs, formatted to 2 decimal places

print(f"Wavelength range: {wavelength[0]} - {wavelength[-1]}")
print(f"Intensity range: {intensity[0]:.2f} - {intensity[-1]:.2f}")


#### Load and Combine all the data

In [ ]:
# Debug: Test glob pattern
test_sensor = "Sensor 2"
test_pattern = os.path.join(PROBES_DIR, f"{test_sensor} Sample*.csv")
print(f"Test pattern: {test_pattern}")
test_files = glob.glob(test_pattern)
print(f"Files found: {test_files}")
print(f"Number of files: {len(test_files)}")

In [ ]:
PROBE_LIST = ["P.1", "P.2"]
# Manual order, because Sensor 2 has the lowest wavelength
SENSOR_LIST = ["Sensor 2", "Sensor 1", "Sensor 4", "Sensor 3"]
get_reference_sample = False

all_sensor_data = {}
for sensor in SENSOR_LIST:
    # Initialize the sensor data list, to store the intensity data of each sample seperately
    sensor_data = []
    sample_paths = get_csv_paths(PROBES_DIR, sensor, get_reference_sample)
    print(f"Debug: Processing {sensor}, found {len(sample_paths)} files")
    for csv_path in sample_paths:
        _, intensity = read_csv_data(csv_path)
        sensor_data.append(intensity)

    # Combine the intensity data of all samples into a single array
    if len(sensor_data) > 0:
        all_sensor_data[sensor] = np.vstack(sensor_data)
        print(f"Sensor Data Shape of {sensor}: {np.shape(all_sensor_data[sensor])} ({len(sample_paths)} samples)") 
        # Print the shape of the combined sensor data array, which should be (number of samples, number of wavelengths)
    else:
        print(f"Warning: No data found for {sensor}")

### Plotting

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def parse_csv_data(path):
    df = pd.read_csv(path, encoding="ISO-8859-1")
    df = df.drop(columns=df.columns[0])
    return df.mean(axis=0).to_numpy(dtype=float)

def get_wavelength_range(sensor):
    if sensor == "Sensor 1":
        return range(1350, 1652, 2)
    if sensor == "Sensor 2":
        return range(1100, 1352, 2)
    if sensor == "Sensor 3":
        return range(1750, 2152, 2)
    if sensor == "Sensor 4":
        return range(1550, 1952, 2)

def combine_o1_o2_all_sensors(folder="SAMPLE O"):
    files = glob.glob(os.path.join(folder, "Sensor * Sample O.* *.csv"))
    groups = {}

    for path in files:
        name = os.path.basename(path)
        match = re.match(r"(Sensor \d+) Sample O\.(\d+) (.+)\.csv", name)

        if not match:
            continue

        sensor = match.group(1)
        replicate = match.group(2)
        condition = match.group(3)

        groups.setdefault((sensor, condition), {})[replicate] = path

    combined = {}

    for key, reps in groups.items():
        if "1" in reps and "2" in reps:
            o1 = parse_csv_data(reps["1"])
            o2 = parse_csv_data(reps["2"])
            combined[key] = np.mean([o1, o2], axis=0)

    return combined


In [ ]:
combined_o = combine_o1_o2_all_sensors("SAMPLE O")

conditions = ["Baseline", "10 MIN", "20 MIN", "30 MIN", "40 MIN"]

fig, axes = plt.subplots(2, 2, figsize=(10, 6), dpi=150)
axes = axes.ravel()

for i, sensor in enumerate(["Sensor 1", "Sensor 2", "Sensor 3", "Sensor 4"]):
    wavelength = list(get_wavelength_range(sensor))

    for condition in conditions:
        intensity = combined_o[(sensor, condition)]
        axes[i].plot(wavelength, intensity, label=condition)

    axes[i].set_title(sensor)
    axes[i].set_xlabel("Wavelength (nm)")
    axes[i].set_ylabel("Intensity")
    axes[i].grid(True, linestyle="--", linewidth=0.5)
    axes[i].legend(fontsize=7)

fig.suptitle("Sample O - Ammonia air stripping over time", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean
import pandas as pd

# Calculate similarity metrics for all sensor data
similarity_results = {}

for sensor in SENSOR_LIST:
    if sensor in all_sensor_data:# Check if the sensor data exists in the all_sensor_data dictionary
        data = all_sensor_data[sensor]
        
        # Cosine similarity between samples
        cosine_sim = cosine_similarity(data)
        
        # Euclidean distance between samples
        euclidean_dist = np.array([[euclidean(data[i], data[j]) 
                                     for j in range(len(data))] 
                                    for i in range(len(data))])
        
        similarity_results[sensor] = {
            'cosine_similarity': cosine_sim,
            'euclidean_distance': euclidean_dist
        }
        
        print(f"\n{sensor} - Cosine Similarity Matrix:")
        print(cosine_sim)
        print(f"\n{sensor} - Euclidean Distance Matrix:")
        print(euclidean_dist)

# Interpretation of Similarity Metrics

This cell calculates two complementary metrics to compare how similar the spectral measurements are between different samples within each sensor. Here's what each means:

## Cosine Similarity

**Range**: -1 to +1 (typically 0 to 1 for spectral data)

- **Value = 1.0**: Samples have identical spectral shape/pattern, though intensity magnitudes may differ
- **Value = 0.9+**: Very similar spectra - good reproducibility
- **Value = 0.5-0.9**: Moderate similarity - detectable differences in spectral features
- **Value < 0.5**: Significantly different spectral patterns

**What it measures**: The angle between two spectral vectors. It's **shape-focused** - it ignores absolute intensity differences and only cares about the relative pattern. This is useful for detecting if samples have the same composition regardless of measurement conditions.

**Example**: Two measurements with proportionally identical peaks but different baseline intensities would still have cosine similarity ≈ 1.0.

## Euclidean Distance

**Range**: 0 to infinity (no fixed upper bound)

- **Value = 0**: Identical spectra at every wavelength
- **Small values (0-100)**: Very similar samples - good measurement reproducibility
- **Medium values (100-500)**: Noticeable differences in intensity or shape
- **Large values (>500)**: Significantly different spectra

**What it measures**: The straight-line distance in multi-dimensional space (one dimension per wavelength). It's **magnitude-sensitive** - it accounts for both shape AND intensity differences. This is useful for detecting actual changes in the measured signal.

**Example**: Two measurements where one has consistently higher intensity will have a larger Euclidean distance, even if they have the same shape.

## Practical Interpretation for Your Data

**For biogas plant monitoring:**

- **High cosine similarity + Low Euclidean distance** → Same composition, consistent measurements ✓
- **High cosine similarity + High Euclidean distance** → Same composition but different concentrations (possibly due to dilution or instrumental drift)
- **Low cosine similarity + Low Euclidean distance** → Noisy data, needs checking
- **Low cosine similarity + High Euclidean distance** → Different chemical composition detected ⚠️

**Matrix structure**: The diagonal should be all 1.0 (cosine) or 0 (Euclidean) since each sample is identical to itself. Off-diagonal values show similarity between different samples - **identical values indicate reproducible, consistent measurements**.

### Statistics 

#### Maximum and Minimum Value
The intensity value of the sensor output should be further normalized based on the data of lamp on/off
